# 03 – Model Development and Hyperparameter Tuning
Module owner: Alix Drouin

Goal: compare several models with the logistic regression baseline, tune their hyperparameters, choose a decision threshold that reaches our recall target (≥ 95 % on phishing), and analyse the errors.

Rules followed in this notebook:
- only `train.csv` is used; the test set stays untouched until the final evaluation (week 4);
- every model is tuned with 5-fold stratified cross-validation on the training set;
- every model is inside a `Pipeline`, so any preprocessing is fitted on the training folds only.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (precision_recall_curve, recall_score, precision_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)
DATA_DIR = Path("../data")
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
X_train, y_train = train.drop(columns="class"), train["class"]
print("Train:", X_train.shape, "| phishing share:", round(y_train.mean(), 3))

## 1. Models and hyperparameter grids
- **Logistic regression**: the baseline, now with a tuned regularisation `C`.
- **Decision tree**: simple non-linear model, easy to interpret, but tends to overfit.
- **k-NN**: classifies a website like its closest neighbours.
- **SVM (RBF kernel)**: non-linear boundary. Scaling is added for SVM and k-NN, inside the pipeline.
- **Random Forest**: many trees on random subsets → less overfitting than one tree.
- **Gradient Boosting** (`HistGradientBoostingClassifier`): trees built one after the other, each correcting the previous errors.

Grids are kept small so that the notebook runs in a few minutes.

In [ ]:
models = {
    "Logistic regression": (
        Pipeline([("clf", LogisticRegression(max_iter=1000, random_state=SEED))]),
        {"clf__C": [0.01, 0.1, 1, 10]}),
    "Decision tree": (
        Pipeline([("clf", DecisionTreeClassifier(random_state=SEED))]),
        {"clf__max_depth": [3, 5, 10, None], "clf__min_samples_leaf": [1, 5, 20]}),
    "k-NN": (
        Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
        {"clf__n_neighbors": [3, 5, 11, 21], "clf__weights": ["uniform", "distance"]}),
    "SVM (RBF)": (
        Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="rbf", random_state=SEED))]),
        {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01, 0.1]}),
    "Random Forest": (
        Pipeline([("clf", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]),
        {"clf__max_depth": [None, 10, 20], "clf__min_samples_leaf": [1, 3],
         "clf__max_features": ["sqrt", 0.5]}),
    "Gradient Boosting": (
        Pipeline([("clf", HistGradientBoostingClassifier(random_state=SEED))]),
        {"clf__learning_rate": [0.05, 0.1], "clf__max_iter": [200, 400],
         "clf__max_depth": [None, 6], "clf__l2_regularization": [0.0, 1.0]}),
}

## 2. Hyperparameter tuning with cross-validation
For each model, `GridSearchCV` tries every combination with 5-fold stratified CV.
The best combination is chosen on **ROC-AUC**, because it does not depend on the decision threshold (the threshold is chosen later, in section 4).
We also keep the **training score** to detect overfitting (big gap between train and CV score).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"recall": "recall", "precision": "precision", "f1": "f1", "roc_auc": "roc_auc"}

searches, rows = {}, {}
for name, (pipe, grid) in models.items():
    t0 = time.time()
    gs = GridSearchCV(pipe, grid, cv=cv, scoring=scoring, refit="roc_auc",
                      return_train_score=True, n_jobs=-1)
    gs.fit(X_train, y_train)
    searches[name] = gs
    i = gs.best_index_
    r = gs.cv_results_
    rows[name] = {
        "recall":    f"{r['mean_test_recall'][i]:.3f} ± {r['std_test_recall'][i]:.3f}",
        "precision": f"{r['mean_test_precision'][i]:.3f} ± {r['std_test_precision'][i]:.3f}",
        "f1":        f"{r['mean_test_f1'][i]:.3f} ± {r['std_test_f1'][i]:.3f}",
        "roc_auc":   f"{r['mean_test_roc_auc'][i]:.3f} ± {r['std_test_roc_auc'][i]:.3f}",
        "train_auc - cv_auc": round(r['mean_train_roc_auc'][i] - r['mean_test_roc_auc'][i], 3),
        "time (s)": round(time.time() - t0, 1),
    }
    print(f"{name:20s} best params: {gs.best_params_}")

results = pd.DataFrame(rows).T
results

**Note:** the scores of the best combination are slightly optimistic, because the same CV folds are used to choose the hyperparameters and to measure the score. The real, unbiased estimate will be the test set (week 4).

In [ ]:
auc = pd.Series({n: s.best_score_ for n, s in searches.items()}).sort_values()
ax = auc.plot.barh(color=["tab:gray" if n == "Logistic regression" else "tab:green" for n in auc.index])
ax.set_xlim(auc.min() - 0.01, 1.0)
ax.set_xlabel("Cross-validated ROC-AUC (best hyperparameters)")
ax.set_title("Model comparison (grey = baseline)")
plt.tight_layout(); plt.savefig(FIG_DIR / "models_roc_auc.png", dpi=150); plt.show()

## 3. Overfitting check
`train_auc - cv_auc` is the gap between the score on the training folds and on the validation folds. A large gap means the model learns the training data by heart.

In [ ]:
gap = results["train_auc - cv_auc"].astype(float).sort_values()
print(gap)

## 4. Decision threshold for the recall target
By default a model predicts "phishing" when the probability is ≥ 0.5. To reach **recall ≥ 95 %**, we can lower this threshold.
To choose it without touching the test set, we use **out-of-fold probabilities** (`cross_val_predict`): each website gets a probability from a model that did not see it during training.
We pick the threshold that gives the **best precision among those with recall ≥ 95 %**.

We do this for the best model and for the baseline, to compare them at the same recall.

In [ ]:
best_name = max(searches, key=lambda n: searches[n].best_score_)
print("Best model:", best_name)

cv_thr = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)   # different folds than tuning

def pick_threshold(estimator, target_recall=0.95):
    proba = cross_val_predict(estimator, X_train, y_train, cv=cv_thr, method="predict_proba")[:, 1]
    prec, rec, thr = precision_recall_curve(y_train, proba)
    prec, rec = prec[:-1], rec[:-1]               # last point has no threshold
    ok = rec >= target_recall
    best = np.argmax(np.where(ok, prec, -1))
    return proba, thr[best], prec, rec, thr

thresholds = {}
for name in [best_name, "Logistic regression"]:
    proba, t, prec, rec, thr = pick_threshold(searches[name].best_estimator_)
    y_hat = (proba >= t).astype(int)
    thresholds[name] = {"proba": proba, "threshold": t}
    print(f"{name:20s} threshold = {t:.3f} | recall = {recall_score(y_train, y_hat):.3f} "
          f"| precision = {precision_score(y_train, y_hat):.3f} | F1 = {f1_score(y_train, y_hat):.3f}")
    plt.plot(rec, prec, label=name)

plt.axvline(0.95, ls="--", color="red", label="recall target 95 %")
plt.xlabel("Recall (phishing)"); plt.ylabel("Precision"); plt.xlim(0.8, 1.0); plt.ylim(0.7, 1.0)
plt.title("Precision–recall curves (out-of-fold, train set)"); plt.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "precision_recall_threshold.png", dpi=150); plt.show()

## 5. Error analysis of the best model
Confusion matrix with the chosen threshold, then: what do the missed phishing websites (false negatives) look like?

In [ ]:
proba = thresholds[best_name]["proba"]
t = thresholds[best_name]["threshold"]
y_hat = (proba >= t).astype(int)

cm = confusion_matrix(y_train, y_hat)
ConfusionMatrixDisplay(cm, display_labels=["legitimate", "phishing"]).plot(cmap="Blues")
plt.title(f"{best_name} – threshold {t:.2f} (out-of-fold, train)")
plt.tight_layout(); plt.savefig(FIG_DIR / "best_model_confusion_matrix.png", dpi=150); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Missed phishing (FN): {fn} | False alarms (FP): {fp}")

In [ ]:
fn_mask = (y_train == 1) & (y_hat == 0)
tp_mask = (y_train == 1) & (y_hat == 1)
compare = pd.DataFrame({
    "missed phishing (FN)": X_train[fn_mask].mean(),
    "detected phishing (TP)": X_train[tp_mask].mean(),
})
compare["difference"] = compare.iloc[:, 0] - compare.iloc[:, 1]
compare.sort_values("difference", key=abs, ascending=False).head(8).round(2)

**How to read it:** features where missed phishing websites have a much higher mean than detected ones are the "legitimate signs" these websites manage to show (e.g. HTTPS or normal anchors). These phishing websites look legitimate on the features we have, which is a limit of the dataset.

## Conclusions
- All models are compared with the same CV folds and the same metrics as the baseline.
- Gradient Boosting is the best model (CV ROC-AUC ≈ 0.993 vs 0.976 for the baseline), followed by Random Forest and SVM. Tree ensembles capture interactions between features that a linear model cannot.
- No model overfits strongly: the train–CV gap stays around 0.01 ROC-AUC or less.
- At the same recall (95 %), Gradient Boosting reaches a precision of ≈ 0.95, against ≈ 0.88 for the logistic regression: far fewer false alarms.
- Missed phishing websites mostly look legitimate on HTTPS and AnchorURL, the two most important features: this is a limit of the dataset, not only of the model.
- The test set has still not been used. Next step (week 4): retrain the chosen model with its threshold on the whole training set and evaluate it once on the test set.